In [3]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

In [55]:
m = gp.Model("MIP")

from ranDataGen.order_data_loader import load_order_data

df = load_order_data(base_dir='ranDataGen/')

# df[df['order'] == 1]           
#df[df['tote'] == 1]            
# df.groupby('order').sum()      
# df.groupby('tote')['quantity'].sum()
df

,order,item_slot,itemtype,quantity,tote
0,1,0,3,3,1
1,1,1,1,2,1
2,2,0,2,3,2
3,2,1,3,1,3
4,2,2,0,1,2
5,3,0,3,3,3
6,3,1,2,3,2
7,3,2,0,1,1
8,4,0,1,1,0
9,4,1,2,1,0


In [58]:
# ==================================================
# Build inventory items: (item_id, item_type, tote)
# ==================================================

items = []
tote_items = {}

item_id = 0

for _, row in df.iterrows():
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        item_id += 1
        item = (item_id, i, t)
        items.append(item)
        
        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

N = len(items)
positions = range(1, N+1)

# ==================================================
# Build demand dictionary
# ==================================================

orders = df["order"].unique()

demand = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    qty = int(row["quantity"])
    
    demand[(o,i)] = demand.get((o,i),0) + qty

# ==================================================
# Item position on conveyor
# ==================================================

x = m.addVars(
    [(item_id,p) for (item_id,i,t) in items for p in positions],
    vtype=GRB.BINARY,
    name="x"
)

# Each item gets exactly one position

m.addConstrs(
    gp.quicksum(x[item_id,p] for p in positions) == 1
    for (item_id,i,t) in items
)

# Each position gets exactly one item

m.addConstrs(
    gp.quicksum(x[item_id,p] for (item_id,i,t) in items) == 1
    for p in positions
)

belts = range(1,5)

y = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign"
)

# each belt gets one order
m.addConstrs(
    gp.quicksum(y[b,o] for o in orders) == 1
    for b in belts
)

# each order goes to one belt
m.addConstrs(
    gp.quicksum(y[b,o] for b in belts) == 1
    for o in orders
)

pick = m.addVars(
    [(b,item_id,o,p) for b in belts
                      for (item_id,i,t) in items
                      for o in orders
                      for p in positions],
    vtype=GRB.BINARY,
    name="pick"
)

m.addConstrs(
    pick[b,item_id,o,p] <= x[item_id,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,item_id,o,p] for b in belts for o in orders) <= 1
    for (item_id,i,t) in items
    for p in positions
)

m.addConstrs(
    pick[b+1,item_id,o,p] <=
    1 - gp.quicksum(pick[b,item_id,o2,p] for o2 in orders)
    
    for b in range(1,4)
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.addConstrs(
    pick[b,item_id,o,p] <= y[b,o]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

for (item_id,i,t) in items:
    for o in orders:
        if (o,i) not in demand:
            for b in belts:
                for p in positions:
                    m.addConstr(pick[b,item_id,o,p] == 0)

for (o,i),qty in demand.items():

    m.addConstr(
        gp.quicksum(
            pick[b,item_id,o,p]
            for b in belts
            for (item_id,it,t) in items if it == i
            for p in positions
        ) == qty
    )

T = m.addVar(vtype=GRB.INTEGER, name="makespan")

m.addConstrs(
    T >= p * pick[b,item_id,o,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.setObjective(T, GRB.MINIMIZE)

m.optimize()

print("\nITEM SEQUENCE ON CONVEYOR")

sequence = []

for (item_id,i,t) in items:
    for p in positions:
        if x[item_id,p].X > 0.5:
            sequence.append((p,item_id,i,t))

sequence.sort()

for p,item_id,i,t in sequence:
    print(f"Position {p}: Item {item_id} | Type {i} | Tote {t}")

print("\nTOTE SEQUENCE")

for p,item_id,i,t in sequence:
    print(f"Position {p}: Tote {t}")

print("\nBELT → ORDER ASSIGNMENTS")

for b in belts:
    for o in orders:
        if y[b,o].X > 0.5:
            print(f"Belt {b} processes Order {o}")

print("\nPICKS")

for b in belts:
    for (item_id,i,t) in items:
        for o in orders:
            for p in positions:
                if pick[b,item_id,o,p].X > 0.5:
                    print(
                        f"Belt {b} picks Item {item_id} "
                        f"(Type {i}, Tote {t}) "
                        f"for Order {o} at position {p}"
                    )

print("\nSYSTEM COMPLETION POSITION:", T.X)

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 23901 rows, 6154 columns and 68622 nonzeros
Model fingerprint: 0xf3f5a0ae
Variable types: 0 continuous, 6154 integer (6153 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]
Found heuristic solution: objective 19.0000000
Presolve removed 19973 rows and 1824 columns
Presolve time: 0.71s
Presolved: 3928 rows, 4330 columns, 20226 nonzeros
Variable types: 0 continuous, 4330 integer (4329 binary)

Root relaxation: infeasible, 1886 iterations, 0.36 seconds (0.19 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 infeas

In [ ]:
# ==================================================
# Build Item, Tote, and Order Structures
# ==================================================

items = []
tote_items = {}
order_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])

    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)

        tote_items.setdefault(t, []).append(item)
        order_items.setdefault(o, []).append(item)

# --------------------------------------------------
# Basic Sets
# --------------------------------------------------

N = len(items)
positions = range(1, N + 1)

orders = sorted(df["order"].unique())
item_types = sorted(df["itemtype"].unique())
belts = [1, 2, 3, 4]

# ==================================================
# Assignment Variables
# ==================================================

x = m.addVars(
    [(o, i, q, t, p) for (o, i, q, t) in items for p in positions],
    vtype=GRB.BINARY
)

# One position per item
m.addConstrs(
    gp.quicksum(x[o, i, q, t, p] for p in positions) == 1
    for (o, i, q, t) in items
)

# One item per position
m.addConstrs(
    gp.quicksum(x[o, i, q, t, p] for (o, i, q, t) in items) == 1
    for p in positions
)

# ==================================================
# Position Mapping
# ==================================================

P = m.addVars(items, vtype=GRB.INTEGER, lb=1, ub=N)

m.addConstrs(
    P[o, i, q, t] ==
    gp.quicksum(p * x[o, i, q, t, p] for p in positions)
    for (o, i, q, t) in items
)

# --------------------------------------------------
# Loading Time Mapping
# --------------------------------------------------

T = m.addVars(items, vtype=GRB.CONTINUOUS, lb=0)

m.addConstrs(
    T[o, i, q, t] ==
    gp.quicksum(3 * (p - 1) * x[o, i, q, t, p] for p in positions)
    for (o, i, q, t) in items
)

# ==================================================
# Tote Contiguity Constraint
# ==================================================

for t in tote_items:

    tote_set = tote_items[t]
    k = len(tote_set)

    if k <= 1:
        continue

    minP = m.addVar(vtype=GRB.INTEGER)
    maxP = m.addVar(vtype=GRB.INTEGER)

    m.addConstrs(minP <= P[o, i, q, t] for (o, i, q, t) in tote_set)
    m.addConstrs(maxP >= P[o, i, q, t] for (o, i, q, t) in tote_set)

    # m.addConstr(maxP - minP == k - 1)

    m.addConstr(maxP - minP <= k)

# ==================================================
# Belt Assignment Variables
# ==================================================

z = m.addVars(
    [(o, b) for o in orders for b in belts],
    vtype=GRB.BINARY
)

# Each order assigned to exactly one belt
m.addConstrs(
    gp.quicksum(z[o, b] for b in belts) == 1
    for o in orders
)

# ==================================================
# Belt Priority Dominance Constraint
# ==================================================

Q = m.addVars(
    [(o, i, b) for o in orders
             for i in item_types
             for b in belts],
    vtype=GRB.INTEGER,
    name="Q"
)

# Upstream dominance flow constraint
# for o in orders:
#     for i in item_types:
        # for b in range(1, len(belts)):
        #     m.addConstr(
        #         Q[o, i, b] >= Q[o, i, b + 1]
        #     )


# Demand aggregation
demand = df.groupby(["order", "itemtype"])["quantity"].sum().to_dict()

m.addConstrs(
    gp.quicksum(Q[o, i, b] for b in belts) ==
    demand.get((o, i), 0)
    for o in orders
    for i in item_types
)

m.addConstrs(
    Q[o, i, b] <= demand.get((o, i), 0) # * z[o, b]
    for o in orders
    for i in item_types
    for b in belts
)

m.addConstrs(
    Q[o,i,b] >= 0
    for o in orders
    for i in item_types
    for b in belts
)

# m.addConstrs(
#     Q[o,i,b] <= demand.get((o,i),0)
#     for o in orders
#     for i in item_types
#     for b in belts
# )

# ==================================================
# Time Coordination Constraint
# ==================================================

PickupTime = m.addVars(items, vtype=GRB.CONTINUOUS)

m.addConstrs(
    PickupTime[o, i, q, t] >=
    gp.quicksum(3 * (p - 1) * x[o, i, q, t, p] for p in positions)
    +
    gp.quicksum((3 + 7.5 * (b - 1)) * z[o, b] for b in belts)
    for (o, i, q, t) in items
)

# ==================================================
# Objective Function
# ==================================================

Cmax = m.addVar(vtype=GRB.CONTINUOUS, name="Cmax")

m.addConstrs(
    Cmax >= PickupTime[u]
    for u in items
)

m.setObjective(Cmax, GRB.MINIMIZE)

In [43]:
# ==================================================
# Build Item, Tote, and Order Structures
# ==================================================

items = []
tote_items = {}
order_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])

    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)

        tote_items.setdefault(t, []).append(item)
        order_items.setdefault(o, []).append(item)

# --------------------------------------------------
# Basic Sets
# --------------------------------------------------

N = len(items)
positions = range(1, N + 1)

orders = sorted(df["order"].unique())
item_types = sorted(df["itemtype"].unique())
belts = [1, 2, 3, 4]

# ==================================================
# Assignment Variables (Permutation Encoding)
# ==================================================

x = m.addVars(
    [(o,i,q,t,p) for (o,i,q,t) in items for p in positions],
    vtype=GRB.BINARY
)

# One position per item
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for p in positions) == 1
    for (o,i,q,t) in items
)

# One item per position
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for (o,i,q,t) in items) == 1
    for p in positions
)

# --------------------------------------------------
# Position Mapping
# --------------------------------------------------

P = m.addVars(items, vtype=GRB.INTEGER, lb=1, ub=N)

m.addConstrs(
    P[o,i,q,t] ==
    gp.quicksum(p * x[o,i,q,t,p] for p in positions)
    for (o,i,q,t) in items
)

# --------------------------------------------------
# Loading Time Mapping (3 sec spacing)
# --------------------------------------------------

load_term = m.addVars(items, lb=0)

m.addConstrs(
    load_term[o,i,q,t] ==
    gp.quicksum(3*(p-1)*x[o,i,q,t,p] for p in positions)
    for (o,i,q,t) in items
)

# ==================================================
# Tote Contiguity Envelope Constraint
# ==================================================

for t in tote_items:

    tote_set = tote_items[t]
    k = len(tote_set)

    if k <= 1:
        continue

    minP = m.addVar(vtype=GRB.INTEGER)
    maxP = m.addVar(vtype=GRB.INTEGER)

    M = N + 10

    m.addConstrs(minP <= P[u] for u in tote_set)
    m.addConstrs(maxP >= P[u] for u in tote_set)

    m.addConstr(maxP - minP <= k - 1 + M*(1-1))

# ==================================================
# Belt Assignment Variables
# ==================================================

z = m.addVars(
    [(o,b) for o in orders for b in belts],
    vtype=GRB.BINARY
)

# Each order assigned to exactly one belt
m.addConstrs(
    gp.quicksum(z[o,b] for b in belts) == 1
    for o in orders
)

# ==================================================
# Demand Aggregation Flow Variables
# ==================================================

Q = m.addVars(
    [(o,i,b) for o in orders
             for i in item_types
             for b in belts],
    vtype=GRB.INTEGER
)

demand = df.groupby(["order","itemtype"])["quantity"].sum().to_dict()

# Demand conservation
m.addConstrs(
    gp.quicksum(Q[o,i,b] for b in belts)
    == demand.get((o,i),0)
    for o in orders
    for i in item_types
)

# Nonnegativity
m.addConstrs(
    Q[o,i,b] >= 0
    for o in orders
    for i in item_types
    for b in belts
)

# Upper bound capacity
m.addConstrs(
    Q[o,i,b] <= demand.get((o,i),0)
    for o in orders
    for i in item_types
    for b in belts
)

# ==================================================
# Pickup Timing Model (Slack Formulation)
# ==================================================

# PickupTime = m.addVars(items, lb=0)

# S = m.addVars(items, lb=0)   # slack variable

# belt_cost = {b: 3 + 7.5*(b-1) for b in belts}

# m.addConstrs(
#     PickupTime[o,i,q,t] ==
#     load_term[o,i,q,t] +
#     gp.quicksum(belt_cost[b]*z[o,b] for b in belts) +
#     S[o,i,q,t]
#     for (o,i,q,t) in items
# )

# ==================================================
# Makespan Objective
# ==================================================

# Cmax = m.addVar(lb=0)

# m.addConstrs(
#     Cmax >= PickupTime[u]
#     for u in items
# )

Cmax = m.addVar(vtype=GRB.CONTINUOUS, name="Cmax")

m.addConstrs(
    Cmax >= P[u]
    for u in items
)


m.setObjective(Cmax, GRB.MINIMIZE)

In [45]:
import gurobipy as gp
from gurobipy import GRB

# ==================================================
# Data Structures
# ==================================================

items = []
tote_items = {}
order_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])

    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)

        tote_items.setdefault(t, []).append(item)
        order_items.setdefault(o, []).append(item)

# ==================================================
# Basic Sets
# ==================================================

N = len(items)
positions = range(1, N + 1)

orders = sorted(df["order"].unique())
item_types = sorted(df["itemtype"].unique())
belts = [1, 2, 3, 4]

# ==================================================
# Model
# ==================================================

m = gp.Model("picker_schedule")

# ==================================================
# Permutation Assignment Encoding
# ==================================================

x = m.addVars(
    [(o,i,q,t,p) for (o,i,q,t) in items for p in positions],
    vtype=GRB.BINARY
)

# Each item gets exactly one position
m.addConstrs(
    gp.quicksum(x[u,p] for p in positions) == 1
    for u in items
)

# Each position gets exactly one item
m.addConstrs(
    gp.quicksum(x[u,p] for u in items) == 1
    for p in positions
)

# --------------------------------------------------
# Position Mapping
# --------------------------------------------------

P = m.addVars(items, lb=1, ub=N, vtype=GRB.INTEGER)

m.addConstrs(
    P[u] == gp.quicksum(p*x[u,p] for p in positions)
    for u in items
)

# --------------------------------------------------
# Loading base time term
# 3 second spacing
# --------------------------------------------------

load_term = m.addVars(items, lb=0)

m.addConstrs(
    load_term[u] ==
    gp.quicksum(3*(p-1)*x[u,p] for p in positions)
    for u in items
)

# ==================================================
# Tote Contiguity Envelope Constraint
# ==================================================

M = N

for t in tote_items:

    block = tote_items[t]
    k = len(block)

    if k <= 1:
        continue

    minP = m.addVar(lb=1, ub=N)
    maxP = m.addVar(lb=1, ub=N)

    for u in block:
        m.addConstr(P[u] >= minP)
        m.addConstr(P[u] <= maxP)

    # block width relaxation
    m.addConstr(maxP - minP <= k - 1 + M*(1-1))

# ==================================================
# Objective Variable
# ==================================================

Cmax = m.addVar(lb=0)

m.addConstrs(
    Cmax >= P[u]
    for u in items
)

m.setObjective(Cmax, GRB.MINIMIZE)

# ==================================================
# Solver Parameters (Important)
# ==================================================

m.Params.MIPFocus = 1
m.Params.Presolve = 2
m.Params.Method = 2

# ==================================================
# Solve
# ==================================================
m.computeIIS()
m.write("model_iis.ilp")
m.optimize()

# ==================================================
# Display Solution
# ==================================================

if m.status == GRB.OPTIMAL:

    solution = sorted(
        [(u, P[u].X) for u in items],
        key=lambda x: x[1]
    )

    print("\n=== Item Sequence ===")
    for u, pos in solution:
        print(u, "-> position", int(pos))

    print("\nMakespan:", Cmax.X)

else:
    print("Model not optimal. Status:", m.status)

KeyError: ((1, 3, 0, 1), 1)

In [44]:
m.computeIIS()
m.write("model_iis.ilp")

m.optimize()

if m.status == GRB.INFEASIBLE:
    print("\nModel is infeasible. Computing IIS...\n")
    m.computeIIS()
    m.write("model_infeasible.ilp")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

IIS computation: initial model status unknown, solving to determine model status
Presolve removed 7519 rows and 9545 columns
Presolve time: 0.18s

Explored 0 nodes (0 simplex iterations) in 0.21 seconds (0.08 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -

Computing Irreducible Inconsistent Subsystem (IIS)...

           Constraints          |            Bounds           |  Runtime
      Min       Max     Guess   |   Min       Max     Guess   |
--------------------------------------------------------------------------
        0      8301         -         0      3149         -           0s
       33        33        33         0         0         0           4s

IIS computed: 33 constr

In [14]:
# ==================================================
# Solve Model
# ==================================================

m.optimize()

if m.status == GRB.INFEASIBLE:
    print("\nModel is infeasible. Computing IIS...\n")
    m.computeIIS()
    m.write("model_infeasible.ilp")

# --------------------------------------------------
# Check Optimization Status
# --------------------------------------------------

if m.status != GRB.OPTIMAL:
    print("Model did not solve to optimality.")
else:
    print("\n===== OPTIMAL SOLUTION FOUND =====\n")

    # ==================================================
    # Extract Belt Assignment Results
    # ==================================================

    # Order → Belt mapping
    order_belt_map = {}

    for o in orders:
        for b in belts:
            if z[o, b].X > 0.5:
                order_belt_map[o] = b

    # ==================================================
    # Build Sequence Loading Lists Per Belt
    # ==================================================

    belt_pick_lists = {b: [] for b in belts}

    # Sort items by sequence position
    sorted_items = sorted(
        [(P[u].X, u) for u in items],
        key=lambda x: x[0]
    )

    # Assign items to belt pick lists
    for _, item in sorted_items:

        o, i, q, t = item

        belt = order_belt_map.get(o, None)

        belt_pick_lists[belt].append({
            "order": o,
            "itemtype": i,
            "quantity_index": q,
            "tote": t,
            "sequence_position": P[item].X,
            "pickup_time": PickupTime[item].X
        })

    # ==================================================
    # Display Results
    # ==================================================

    for b in belts:
        print(f"\n------------------------------")
        print(f"Belt {b} Pick List (Loading Order)")
        print(f"------------------------------")

        pick_list = sorted(
            belt_pick_lists[b],
            key=lambda x: x["sequence_position"]
        )

        for entry in pick_list:
            print(
                f"Order {entry['order']} | "
                f"ItemType {entry['itemtype']} | "
                f"QtyIndex {entry['quantity_index']} | "
                f"Tote {entry['tote']} | "
                f"SeqPos {entry['sequence_position']:.0f} | "
                f"Time {entry['pickup_time']:.2f}s"
            )

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2041 rows, 3506 columns and 15059 nonzeros
Model fingerprint: 0x5f149f01
Variable types: 214 continuous, 3292 integer (2687 binary)
Coefficient statistics:
  Matrix range     [1e-01, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 2e+01]
Presolve removed 1544 rows and 2414 columns
Presolve time: 0.14s

Explored 0 nodes (0 simplex iterations) in 0.15 seconds (0.04 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -

Model is infeasible. Computing IIS...

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical 

In [17]:
# total quantity
total_qty = df["quantity"].sum()

# adding sequence variable
S_index = []

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        S_index.append((o, i, q))

S = m.addVars(S_index, vtype=GRB.INTEGER, name="S")


In [ ]:
# build items per order, type, quantity, and tote
items = []
tote_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)
        
        # build tote dictionary
        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

N = len(items)   # should be 19 for demo
positions = range(1, N+1)

x = m.addVars(
    [(o,i,q,t,p) for (o,i,q,t) in items for p in positions],
    vtype=GRB.BINARY,
    name="x"
)

# Each item gets exactly one position
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for p in positions) == 1
    for (o,i,q,t) in items
)

# Each position gets exactly one item
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for (o,i,q,t) in items) == 1
    for p in positions
)

# ______________________________________________________________

belts = range(1,5)

pick = m.addVars(
    [(b,o,i,q,t,p) for b in belts
                   for (o,i,q,t) in items
                   for p in positions],
    vtype=GRB.BINARY,
    name="pick"
)

m.addConstrs(
    pick[b,o,i,q,t,p] <= x[o,i,q,t,p]
    for b in belts
    for (o,i,q,t) in items
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,o,i,q,t,p] for b in belts) <= 1
    for (o,i,q,t) in items
    for p in positions
)

m.addConstrs(
    pick[b+1,o,i,q,t,p] <= 1 - pick[b,o,i,q,t,p]
    for b in range(1,4)
    for (o,i,q,t) in items
    for p in positions
)

orders = df["order"].unique()

y = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign"
)

m.addConstrs(
    gp.quicksum(y[b,o] for o in orders) == 1
    for b in belts
)

m.addConstrs(
    gp.quicksum(y[b,o] for b in belts) == 1
    for o in orders
)

m.addConstrs(
    pick[b,o,i,q,t,p] <= y[b,o]
    for b in belts
    for (o,i,q,t) in items
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,o,i,q,t,p]
                for b in belts
                for p in positions) == 1
    for (o,i,q,t) in items
)

In [48]:
belts = [1,2,3,4]
orders = df["order"].unique()

z = m.addVars(
    [(o,b) for o in orders for b in belts],
    vtype=GRB.BINARY,
    name="z"
)

# Each order gets exactly one belt
m.addConstrs(
    gp.quicksum(z[o,b] for b in belts) == 1
    for o in orders
)

B = m.addVars(
    [(o,b) for o in orders for b in belts],
    vtype=GRB.INTEGER,
    lb=0,
    name="B"
)

order_index = {o:i for i,o in enumerate(sorted(orders))}

m.addConstrs(
    B[o,b] ==
    gp.quicksum(
        order_index[o2] * z[o2,b]
        for o2 in orders
    )
    for o in orders
    for b in belts
)

order_items = {}

for (o,i,q,t) in items:
    if o not in order_items:
        order_items[o] = []
    order_items[o].append((o,i,q,t))

start = m.addVars(orders, belts, vtype=GRB.INTEGER)
end = m.addVars(orders, belts, vtype=GRB.INTEGER)

for o in orders:
    for b in belts:
        m.addConstrs(
            start[o,b] <= P[u]
            for o in orders
            for b in belts
            for u in order_items[o]
        )

for o in orders:
    for b in belts:
        m.addConstrs(
            start[o,b] <= P[u]
            for o in orders
            for b in belts
            for u in order_items[o]
        )

for b in belts:
    y = m.addVars(
        [(o1,o2,b)
        for o1 in orders
        for o2 in orders
        if o1 != o2
        for b in belts],
        vtype=GRB.BINARY
    )

GurobiError: Variable not in model

In [10]:
# --------------------------------------------------
# Build item and tote structures
# --------------------------------------------------

items = []
tote_items = {}
order_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])

    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)

        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

        if o not in order_items:
            order_items[o] = []
        order_items[o].append(item)

# --------------------------------------------------
# Basic Sets
# --------------------------------------------------

N = len(items)
positions = range(1, N+1)

orders = sorted(df["order"].unique())
item_types = sorted(df["itemtype"].unique())
belts = [1,2,3,4]

# --------------------------------------------------
# Assignment Variables
# --------------------------------------------------

x = m.addVars(
    [(o,i,q,t,p) for (o,i,q,t) in items for p in positions],
    vtype=GRB.BINARY
)

# One position per item
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for p in positions) == 1
    for (o,i,q,t) in items
)

# One item per position
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for (o,i,q,t) in items) == 1
    for p in positions
)

# --------------------------------------------------
# Position Mapping
# --------------------------------------------------

P = m.addVars(items, vtype=GRB.INTEGER, lb=1, ub=N)

m.addConstrs(
    P[o,i,q,t] ==
    gp.quicksum(p * x[o,i,q,t,p] for p in positions)
    for (o,i,q,t) in items
)

# Loading time mapping (3 seconds spacing)
T = m.addVars(items, vtype=GRB.CONTINUOUS, lb=0)

m.addConstrs(
    T[o,i,q,t] ==
    gp.quicksum(3*(p-1) * x[o,i,q,t,p] for p in positions)
    for (o,i,q,t) in items
)

# --------------------------------------------------
# Tote Contiguity Constraint
# --------------------------------------------------

for t in tote_items:

    tote_set = tote_items[t]
    k = len(tote_set)

    if k <= 1:
        continue

    minP = m.addVar(vtype=GRB.INTEGER)
    maxP = m.addVar(vtype=GRB.INTEGER)

    m.addConstrs(minP <= P[o,i,q,t] for (o,i,q,t) in tote_set)
    m.addConstrs(maxP >= P[o,i,q,t] for (o,i,q,t) in tote_set)

    m.addConstr(maxP - minP == k - 1)

# --------------------------------------------------
# Belt Assignment Variables
# --------------------------------------------------

z = m.addVars(
    [(o,b) for o in orders for b in belts],
    vtype=GRB.BINARY
)

# Each order assigned to exactly one belt
m.addConstrs(
    gp.quicksum(z[o,b] for b in belts) == 1
    for o in orders
)

# --------------------------------------------------
# Belt Priority Dominance Constraint
# --------------------------------------------------

item_types = sorted(df["itemtype"].unique())

# Upstream belt dominance rule
# If belt b can process item type i,
# then belt b-1 must have finished its quantity first.

Q = m.addVars(
    [(o,i,b) for o in orders
             for i in item_types
             for b in belts],
    vtype=GRB.INTEGER,
    name="Q"
)

# Flow dominance constraint
for o in orders:
    for i in item_types:
        for b in range(1, len(belts)):
            m.addConstr(
                Q[o,i,b] >= Q[o,i,b+1]
            )

# Link belt assignment to quantity processing
demand = df.groupby(["order","itemtype"])["quantity"].sum().to_dict()

m.addConstrs(
    gp.quicksum(Q[o,i,b] for b in belts) == demand.get((o,i),0)
    for o in orders
    for i in item_types
)

m.addConstrs(
    Q[o,i,b] <= demand.get((o,i),0) * z[o,b]
    for o in orders
    for i in item_types
    for b in belts
)

# --------------------------------------------------
# Time Coordination Constraint
# --------------------------------------------------

PickupTime = m.addVars(items, vtype=GRB.CONTINUOUS)

m.addConstrs(
    PickupTime[o,i,q,t] ==
    gp.quicksum(3*(p-1) * x[o,i,q,t,p] for p in positions)
    +
    gp.quicksum((3 + 7.5*(b-1)) * z[o,b]
                for b in belts)
    for (o,i,q,t) in items
)

# --------------------------------------------------
# Objective Function: Minimize Makespan
# --------------------------------------------------

Cmax = m.addVar(vtype=GRB.CONTINUOUS, name="Cmax")

m.addConstrs(
    Cmax >= PickupTime[u]
    for u in items
)

m.setObjective(Cmax, GRB.MINIMIZE)


In [9]:
# --------------------------------------------------
# Build item and tote structures
# --------------------------------------------------

items = []
tote_items = {}
order_items = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])

    for q in range(qty):
        item = (o, i, q, t)
        items.append(item)

        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

        if o not in order_items:
            order_items[o] = []
        order_items[o].append(item)

# --------------------------------------------------
# Basic Sets
# --------------------------------------------------

N = len(items)
positions = range(1, N+1)

orders = sorted(df["order"].unique())
belts = [1,2,3,4]

# --------------------------------------------------
# Assignment Variables
# --------------------------------------------------

x = m.addVars(
    [(o,i,q,t,p) for (o,i,q,t) in items for p in positions],
    vtype=GRB.BINARY
)

# One position per item
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for p in positions) == 1
    for (o,i,q,t) in items
)

# One item per position
m.addConstrs(
    gp.quicksum(x[o,i,q,t,p] for (o,i,q,t) in items) == 1
    for p in positions
)

# --------------------------------------------------
# Position Mapping
# --------------------------------------------------

P = m.addVars(items, vtype=GRB.INTEGER, lb=1, ub=N)

m.addConstrs(
    P[u] ==
    gp.quicksum(p * x[u[0],u[1],u[2],u[3],p] for p in positions)
    for u in items
)

# Loading time mapping
T = m.addVars(items, vtype=GRB.CONTINUOUS, lb=0)

m.addConstrs(
    T[u] ==
    gp.quicksum(3*(p-1) * x[u[0],u[1],u[2],u[3],p] for p in positions)
    for u in items
)

# --------------------------------------------------
# Tote Contiguity Constraint (True Block Envelope)
# --------------------------------------------------

for t in tote_items:

    tote_set = tote_items[t]
    k = len(tote_set)

    if k <= 1:
        continue

    minP = m.addVar(vtype=GRB.INTEGER)
    maxP = m.addVar(vtype=GRB.INTEGER)

    m.addConstrs(minP <= P[u] for u in tote_set)
    m.addConstrs(maxP >= P[u] for u in tote_set)

    m.addConstr(maxP - minP == k - 1)

# --------------------------------------------------
# Belt Assignment Variables
# --------------------------------------------------

z = m.addVars(
    [(o,b) for o in orders for b in belts],
    vtype=GRB.BINARY
)

# Each order assigned to exactly one belt
m.addConstrs(
    gp.quicksum(z[o,b] for b in belts) == 1
    for o in orders
)


# --------------------------------------------------
# 
# --------------------------------------------------

Q = m.addVars(
    [(o,i,b) for o in orders
                 for i in item_types
                 for b in belts],
    vtype=GRB.INTEGER,
    name="Q"
)

for o in orders:
    for i in item_types:
        for b in range(1, len(belts)):
            m.addConstr(
                Q[o,i,b] >= Q[o,i,b+1]
            )

# --------------------------------------------------
# Basic Test Objective (Makespan)
# --------------------------------------------------

Cmax = m.addVar(vtype=GRB.CONTINUOUS)

m.addConstrs(
    Cmax >= T[u]
    for u in items
)

m.setObjective(Cmax, GRB.MINIMIZE)

In [6]:
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 469 rows, 504 columns and 2365 nonzeros
Model fingerprint: 0xb4bef39f
Variable types: 20 continuous, 484 integer (425 binary)
Coefficient statistics:
  Matrix range     [1e-01, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 2e+01]
Presolve removed 310 rows and 212 columns
Presolve time: 0.02s

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


In [ ]:
m.optimize()

print(m.status)

if m.status != GRB.OPTIMAL:
    print("No optimal solution found")
    exit()

seq = sorted(
    [(P[u].X, u) for u in items],
    key=lambda x: x[0]
)

for pos, item in seq:
    print(pos, item)

order_belt_map = {}

for o in orders:
    for b in belts:
        if z[o,b].X > 0.5:
            order_belt_map[o] = b

belt_items = {b: [] for b in belts}

for u in items:
    o, i, q, t = u

    # Find belt for this order
    belt = order_belt_map[o]

    belt_items[belt].append((P[u].X, u))

for b in belts:
    belt_items[b].sort(key=lambda x: x[0])

for b in belts:
    belt_items[b].sort(key=lambda x: x[0])

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 469 rows, 504 columns and 2365 nonzeros
Model fingerprint: 0xb4bef39f
Variable types: 20 continuous, 484 integer (425 binary)
Coefficient statistics:
  Matrix range     [1e-01, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 2e+01]
Presolve removed 310 rows and 212 columns
Presolve time: 0.02s

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
3
No optimal solution found


AttributeError: Unable to retrieve attribute 'X'

: 

## ---------------------------------------------

In [ ]:
# Derive dimensions from df
n_items = df['itemtype'].max() + 1      # item types (0-indexed in df)
N_total = int(df['quantity'].sum())     # total item units in the system
totes_list = sorted(df['tote'].unique())
n_totes = len(totes_list)               # t indexes 0..n_totes-1; totes_list[t]=tote_id
max_q = int(df['quantity'].max())       # max quantity (for q index)

Svar = m.addVars(N_total, n_items, vtype=GRB.INTEGER, lb=0, name="Svar")

# X_stiq[s,t,i,q] : s=total quantities, t=tote index, i=item types, q=quantity index
X_stiq = m.addVars(N_total, n_totes, n_items, max_q + 1, vtype=GRB.BINARY, lb=0, name="X_stiq")

In [5]:
# Output all Svar variables
svars_list = []
for s in range(N_total):
    for i in range(n_items):
        val = Svar[s, i].X if m.SolCount > 0 else None
        svars_list.append({'s': s, 'i (item type)': i, 'value': val})
svars_df = pd.DataFrame(svars_list)
if m.SolCount > 0:
    svars_pivot = svars_df.pivot(index='s', columns='i (item type)', values='value')
    display(svars_pivot)
else:
    print(svars_df.to_string())

i (item type),0,1,2
s,,,
0,8.0,-0.0,-0.0
1,-0.0,-0.0,-0.0
2,-0.0,-0.0,-0.0
3,-0.0,-0.0,-0.0
4,-0.0,-0.0,-0.0
5,-0.0,-0.0,-0.0
6,-0.0,-0.0,-0.0
7,-0.0,-0.0,-0.0


In [6]:
# Output all X_stiq variables
x_list = []
for s in range(N_total):
    for t in range(n_totes):
        for i in range(n_items):
            for q in range(max_q + 1):
                val = X_stiq[s, t, i, q].X if m.SolCount > 0 else None
                x_list.append({'s': s, 't (tote)': totes_list[t], 'i (item type)': i, 'q': q, 'value': val})
x_df = pd.DataFrame(x_list)
if m.SolCount > 0:
    x_nonzero = x_df[x_df['value'] != 0]
    display(x_nonzero) if len(x_nonzero) > 0 else display(x_df)
else:
    print(x_df.to_string())

,s,t (tote),i (item type),q,value
0,0,1,0,0,1.0
1,0,1,0,1,1.0
2,0,1,0,2,1.0
3,0,1,1,0,1.0
4,0,1,1,1,1.0
...,...,...,...,...,...
211,7,4,1,1,1.0
212,7,4,1,2,1.0
213,7,4,2,0,1.0
214,7,4,2,1,1.0
